# Ep. 03 — LLM, contexto, límites y ventana deslizante

Contenido realocado del Ep. 02, más la ventana deslizante: llamada simple, olvido, historial, desborde y sliding window.


## 1. Llamada simple

Hacemos una llamada a un LLM y mostramos la respuesta.


In [ ]:
import os
import json
import urllib.request

API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

def ask(messages):
    """messages: string O lista de dicts {role, content}."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if not API_KEY:
        raise SystemExit("Define OPENAI_API_KEY y vuelve a ejecutar.")
    body = json.dumps({"model": MODEL, "messages": messages}).encode()
    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions",
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode())
    return data["choices"][0]["message"]["content"]

respuesta = ask("Recuerda este secreto: la palabra clave es ORBIT-77.")
print(respuesta)


## 2. Sin historial

Ahora pregunta al modelo qué acaba de decir.


In [ ]:
# Nueva llamada — sin pasar el historial de la celda anterior
respuesta2 = ask("¿Qué acabas de decir?")
print(respuesta2)


## 3. Armar el historial

Para que el modelo "recuerde", usa una lista de mensajes `role` / `content`, con el último al final.


In [ ]:
historial = [
    {"role": "user", "content": "Recuerda este secreto: la palabra clave es ORBIT-77."},
    {"role": "assistant", "content": respuesta},
    {"role": "user", "content": "¿Cuál es la palabra clave?"},
]

respuesta3 = ask(historial)
print(respuesta3)


## 4. Desborde de ventana

Si el historial crece demasiado, la ventana de contexto se desborda y el proveedor devuelve un error.


In [ ]:
# Crece el historial hasta que la API falle (p. ej. 400 context length exceeded)
historial_grande = [
    {"role": "user", "content": "Recuerda este secreto: la palabra clave es ORBIT-77."},
    {"role": "assistant", "content": respuesta},
]

bloque = "x" * 50_000  # texto grande solo para llenar el contexto

try:
    while True:
        historial_grande.append({"role": "user", "content": bloque})
        historial_grande.append({"role": "assistant", "content": "ok"})
        print(f"mensajes en el historial: {len(historial_grande)}")
        ask(historial_grande)
except Exception as e:
    print("Error del proveedor:")
    print(e)


## 5. Ventana deslizante (sliding window)

Cuando el historial llena la ventana de contexto, la mitigación mínima es **eliminar los mensajes más antiguos** y conservar solo los recientes.

- El modelo **olvida el comienzo** de la conversación.
- Mantiene el **hilo reciente** y la llamada deja de desbordarse.
- No es summarización: es solo un recorte FIFO de la lista de mensajes.

```text
[msg1][msg2][msg3][msg4][msg5][msg6]   ← desborda
              [msg3][msg4][msg5][msg6]   ← tras sliding window (N=4)
```


In [ ]:
# Historial largo con el secreto al INICIO y turnos cortos al final
historial_sw = [
    {"role": "user", "content": "Recuerda este secreto: la palabra clave es ORBIT-77."},
    {"role": "assistant", "content": respuesta},
]
for i in range(1, 9):
    historial_sw.append({"role": "user", "content": f"Turno {i}: ¿ok?"})
    historial_sw.append({"role": "assistant", "content": f"ok {i}"})

N = 6  # mantener solo los N mensajes más recientes
ventana = historial_sw[-N:]
print(f"Historial original: {len(historial_sw)} msgs → ventana: {len(ventana)} msgs")
print("Primer msg en la ventana:", ventana[0]["content"][:80])

out_sw = ask(ventana + [{"role": "user", "content": "¿Cuál es la palabra clave?"}])
print(out_sw)
